# Final Project — Step 4: Export Gold to CSV for Power BI

Reads the 3 Gold Delta tables and exports them as single CSV files.
Power BI Desktop imports these directly — no connector or cloud needed.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("export-gold-for-powerbi")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("✓ Spark session ready")

Spark version: 3.5.0
✓ Spark session ready


In [2]:
# Paths
BASE_PATH        = "/workspace/output/final_project"
GOLD_DAILY_PATH  = f"{BASE_PATH}/gold/daily_metrics"
GOLD_HOURLY_PATH = f"{BASE_PATH}/gold/hourly_patterns"
GOLD_ZONE_PATH   = f"{BASE_PATH}/gold/zone_performance"

# CSV export destination — mounted to host at ./output/final_project/powerbi_export/
EXPORT_PATH = f"{BASE_PATH}/powerbi_export"

print(f"Export destination: {EXPORT_PATH}")

Export destination: /workspace/output/final_project/powerbi_export


In [3]:
# Read all 3 Gold tables
df_daily   = spark.read.format("delta").load(GOLD_DAILY_PATH)
df_hourly  = spark.read.format("delta").load(GOLD_HOURLY_PATH)
df_zone    = spark.read.format("delta").load(GOLD_ZONE_PATH)

print(f"daily_metrics    : {df_daily.count():>3} rows, {len(df_daily.columns)} columns")
print(f"hourly_patterns  : {df_hourly.count():>3} rows, {len(df_hourly.columns)} columns")
print(f"zone_performance : {df_zone.count():>3} rows, {len(df_zone.columns)} columns")

daily_metrics    :  35 rows, 9 columns
hourly_patterns  :  48 rows, 8 columns
zone_performance : 253 rows, 7 columns


In [4]:
# Export to single CSV files (coalesce(1) forces a single output file per table)
# Power BI reads one file per table — not a folder of partitioned Parquet parts

tables = {
    "daily_metrics":    df_daily,
    "hourly_patterns":  df_hourly,
    "zone_performance": df_zone,
}

for name, df in tables.items():
    out_path = f"{EXPORT_PATH}/{name}"
    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(out_path)
    )
    print(f"✓ {name} exported to {out_path}")

✓ daily_metrics exported to /workspace/output/final_project/powerbi_export/daily_metrics
✓ hourly_patterns exported to /workspace/output/final_project/powerbi_export/hourly_patterns
✓ zone_performance exported to /workspace/output/final_project/powerbi_export/zone_performance


In [5]:
# Preview each table — verify data looks correct before opening Power BI

print("=== daily_metrics (top 5) ===")
df_daily.orderBy("pickup_date").show(5, truncate=False)

print("=== hourly_patterns (weekday rush hour) ===")
df_hourly.filter("is_weekend = false").orderBy("pickup_hour").show(10, truncate=False)

print("=== zone_performance (top 10 by revenue) ===")
df_zone.show(10, truncate=False)

=== daily_metrics (top 5) ===
+-----------+-----------+-------------+--------+------------------+----------------+----------+-----------+----------------+
|pickup_date|total_trips|total_revenue|avg_fare|avg_distance_miles|avg_duration_min|total_tips|avg_tip_pct|total_passengers|
+-----------+-----------+-------------+--------+------------------+----------------+----------+-----------+----------------+
|2002-12-31 |1          |10.5         |10.5    |0.63              |6.03            |0.0       |0.0        |1               |
|2009-01-01 |3          |127.69       |42.56   |7.44              |27.62           |0.0       |0.0        |4               |
|2023-12-31 |10         |224.62       |22.46   |2.6               |10.16           |26.27     |18.23      |19              |
|2024-01-01 |67408      |2104286.02   |31.22   |4.32              |16.81           |251088.85 |16.95      |105719          |
|2024-01-02 |70485      |2182468.69   |30.96   |4.19              |17.04           |254941.97 |